# HW — 찍으면 이득인가

**확률통계 · Topic 3** | 배점 25점 | 개인 과제

---

## ✍️ 제출자 정보 — 먼저 채우세요

| | |
|---|---|
| **학번** | (여기에 작성) |

> 파일명을 **`HW_학번_T03.ipynb`** 로 바꿔서 제출한다. (예: `HW_202512345_T03.ipynb`)
> ⚠️ **제출 방법과 기한은 PLATO · Google Classroom 공지**를 확인한다.

---

## 상황

**5지선다 20문항** 시험이 있다. 채점 규칙은 이렇다.

| 결과 | 점수 |
|---|:-:|
| 정답 | **+5** |
| 오답 | **−1** |
| 무응답 | 0 |

아는 문제가 하나도 없어서 **20문항을 전부 찍었다.** 총점을 $X$ 라 하자.

> **찍는 것이 이득인가, 손해인가?** 그리고 그 답을 **얼마나 믿을 수 있는가?**

⭐ 오늘 배운 **선형성**을 쓰면 **총점의 분포를 구하지 않고도** 평균과 분산이 나온다.
그것이 이 과제의 핵심이다.

> **가정** — 보기 다섯 중 정답은 하나이고 무작위로 하나를 고른다 ($p = 1/5$).
> 문항끼리는 **독립**이라고 둔다. 이 가정이 어디서 필요한지는 문제 2 에서 짚는다.

### 할 일

| 문제 | 내용 | 배점 |
|:-:|---|:-:|
| 1 | 한 문항의 PMF · 기댓값 · 제곱의 기댓값 · 분산 | 5 |
| 2 | 선형성으로 20문항으로 확장 + 서술 (c)(d) | 8 |
| 3 | 10만 회 시뮬레이션 · 비교표 · 그래프 둘 | 7 |
| 4 | 해석과 설계 (a)(b)(c) | 5 |

⚠️ **제출 전 `런타임 → 모두 실행`** 으로 출력을 남길 것. 출력이 없으면 −2점.

## Part 0. 준비

이 셀을 먼저 실행한다. **상수와 시드는 바꾸지 말 것** — 채점자가 같은 값을 재현해야 한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(20260302)   # ⚠️ 이 줄은 바꾸지 마세요

N_Q = 20          # 문항 수
P = 1 / 5         # 한 문항을 맞힐 확률
RIGHT, WRONG = 5.0, -1.0

N_TRIALS = 100_000

print(f"{N_Q}문항 · 정답 {RIGHT:+.0f} · 오답 {WRONG:+.0f} · 맞힐 확률 {P:.2f}")

## 문제 1 — 한 문항의 모형 (5점)

한 문항에서 얻는 점수를 $X_i$ 라 하자. 값은 두 가지뿐이다.

### ✏️ TODO 1 — PMF 를 배열 두 개로 적는다

`values` 에 나올 수 있는 점수를, `probs` 에 각각의 확률을 넣는다.

In [ ]:
# TODO 1: 한 문항의 PMF 를 적으세요
#         values - 나올 수 있는 점수 두 가지 (RIGHT, WRONG 을 쓰면 된다)
#         probs  - 각각의 확률 (합이 정확히 1이어야 한다)
values = np.array([0.0, 0.0])      # <- 이 줄을 고치세요
probs = np.array([0.5, 0.5])       # <- 이 줄을 고치세요

assert np.all(probs >= 0), "확률은 음수일 수 없다"
assert abs(probs.sum() - 1.0) < 1e-12, f"확률의 합이 1이 아니다: {probs.sum()}"

print("한 문항의 PMF")
for v, p in zip(values, probs):
    print(f"  P[X_i = {v:+.0f}] = {p:.4f}")
print(f"  합 = {probs.sum():.4f}")

### ✏️ TODO 2 — 네 값을 정의대로 구한다

$$\mathbb{E}[X_i] = \sum_x x\,p(x) \qquad
\mathbb{E}[X_i^2] = \sum_x x^2 p(x) \qquad
\mathrm{Var}[X_i] = \mathbb{E}[X_i^2] - (\mathbb{E}[X_i])^2$$

⚠️ $\mathbb{E}[X_i^2]$ 를 $(\mathbb{E}[X_i])^2$ 로 계산하면 **분산이 0 이 나온다.** 그건 틀렸다.
$g(x) = x^2$ 을 **원래 PMF 에 씌워서** 더하는 것이 LOTUS 다.

In [ ]:
# TODO 2: 아래 네 값을 정의대로 계산하세요
#         힌트 - (values * probs).sum() 이 곧 sum(x * p(x)) 입니다
E1 = 0.0        # <- E[X_i]
E1_sq = 0.0     # <- E[X_i^2]   ⚠️ E1 ** 2 가 아닙니다
Var1 = E1_sq - E1 ** 2
sd1 = Var1 ** 0.5

print(f"  E[X_i]     = {E1:>8.4f}")
print(f"  E[X_i^2]   = {E1_sq:>8.4f}")
print(f"  Var[X_i]   = {Var1:>8.4f}")
print(f"  sigma      = {sd1:>8.4f}")

### 계산 과정을 손으로 적는다 (숫자만 적으면 감점)

위 코드가 한 일을 **식으로** 적는다. 이 셀을 더블클릭해 작성한다.

> **$\mathbb{E}[X_i]$ =** (여기에 작성 — 예: $0.2 \times 5 + 0.8 \times (-1) = \ldots$)
>
> **$\mathbb{E}[X_i^2]$ =** (여기에 작성)
>
> **$\mathrm{Var}[X_i]$ =** (여기에 작성)

## 문제 2 — 20문항으로 확장 (8점)

$X = X_1 + X_2 + \cdots + X_{20}$ 이다.

### ✏️ TODO 3 — 선형성으로 전체 평균과 분산을 구한다

$$\mathbb{E}[X] = 20\,\mathbb{E}[X_i] \qquad
\mathrm{Var}[X] = 20\,\mathrm{Var}[X_i]$$

⚠️ **표준편차는 20배가 아니다.** 분산이 20배이므로 $\sigma$ 는 $\sqrt{20}$ 배다.

In [ ]:
# TODO 3: 선형성으로 20문항의 평균과 분산을 구하세요
#         힌트 - 평균도 N_Q 배, 분산도 N_Q 배. 표준편차는 분산의 제곱근.
EX = 0.0        # <- E[X]
VarX = 0.0      # <- Var[X]
sdX = VarX ** 0.5

print(f"  E[X]     = {EX:>10.4f}")
print(f"  Var[X]   = {VarX:>10.4f}")
print(f"  sigma_X  = {sdX:>10.4f}")
print()
print(f"  참고) sigma 를 그냥 20배 하면 {20 * sd1:.2f} 인데 이는 틀린 값이다")

### (c) 독립 가정은 어디에 필요한가 — 한 문장

위 두 계산 중 **문항끼리 독립이라는 가정이 필요한 것은 어느 쪽인가?**
이 셀을 더블클릭해 작성한다.

> **답:** (여기에 작성)

### (d) 총점의 분포를 구하려면 — 한 문장씩

- 총점 $X$ 의 PMF 를 직접 구하려면 무엇을 해야 하는가?
- **왜 그럴 필요가 없었는가?**

> 💡 총점이 가질 수 있는 값은 $-20$ 부터 $100$ 까지 21가지다.
> 그 21개의 확률을 다 구하는 것과, 조각의 평균을 20번 더하는 것 중 어느 쪽이 쉬운가?

> **답:** (여기에 작성)

## 문제 3 — 시뮬레이션 검증 (7점)

### ✏️ TODO 4 — 10만 번 시험을 친다

한 번의 시험은 **20문항을 각각 확률 $p$ 로 맞히는 것**이다.
`rng.random((N_TRIALS, N_Q)) < P` 를 쓰면 10만 × 20 짜리 True/False 판이 한 번에 나온다.

In [ ]:
# TODO 4: 10만 번의 시험 점수를 만드세요
#         힌트 - hit = rng.random((N_TRIALS, N_Q)) < P
#                score = np.where(hit, RIGHT, WRONG).sum(axis=1)
score = np.zeros(N_TRIALS)      # <- 이 줄을 고치세요

rows = [
    ("E[X]", EX, score.mean()),
    ("Var[X]", VarX, score.var()),
    ("sigma", sdX, score.std()),
]

print(f"{'항목':<10}{'이론값':>14}{'표본값':>14}{'차이':>12}")
print("-" * 52)
for name, theo, emp in rows:
    print(f"{name:<10}{theo:>14.4f}{emp:>14.4f}{emp - theo:>12.4f}")

print()
print(f"  최솟값 {score.min():.0f} · 최댓값 {score.max():.0f}")

### ✏️ TODO 5 — 그림 두 장

왼쪽은 **총점의 히스토그램**(평균 위치에 세로선),
오른쪽은 **누적 평균이 $\mathbb{E}[X]$ 로 수렴하는 그래프**(x축 로그 스케일)다.

> 📌 라벨은 **영어**로. Colab 에는 한글 폰트가 없어 글자가 □□□ 로 깨진다.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))

# TODO 5: 왼쪽에 총점 히스토그램을 그리세요
#         힌트 - ax1.hist(score, bins=np.arange(score.min()-0.5, score.max()+1.5, 3))

ax1.axvline(EX, color="red", ls="--", lw=2, label=f"E[X] = {EX:.1f}")
ax1.axvline(0, color="black", ls=":", lw=1.5, label="zero")
ax1.set_xlabel("Total score")
ax1.set_ylabel("Frequency")
ax1.set_title("Distribution of the total score")
ax1.legend()

n = np.arange(1, N_TRIALS + 1)
running = np.cumsum(score) / n
ax2.plot(n, running, lw=1.2, label="Running average")
ax2.axhline(EX, color="red", ls="--", lw=1.8, label=f"E[X] = {EX:.1f}")
ax2.set_xscale("log")
ax2.set_xlabel("Number of exams (log scale)")
ax2.set_ylabel("Running average")
ax2.set_title("Does it settle at E[X]?")
ax2.legend()
ax2.grid(alpha=0.3, which="both")

plt.tight_layout()
plt.show()

## 문제 4 — 해석과 설계 (5점)

### (a) 찍는 것이 이득인가 — 2~3문장

$\mathbb{E}[X]$ 를 근거로 답한다. 이 셀을 더블클릭해 작성한다.

> **답:** (여기에 작성)

### (b) 공정한 감점은 얼마인가 — 식을 세워서

출제자가 **"찍으면 기댓값이 정확히 0"** 이 되도록 오답 감점을 조정하려 한다.
감점을 $c$ 라 두고 **한 문항의 기댓값이 0 이 되는 식**을 세워 $c$ 를 구한다.

> 💡 $p \times 5 + (1-p) \times (-c) = 0$ 을 $c$ 에 대해 풀면 된다.

> **답:** (여기에 작성 — 식과 값)

### (c) 평균이 양수면 찍어야 하는가

아래 셀을 실행해 **0점 이하가 나온 비율**을 확인한 뒤, 그 숫자와 $\sigma_X$ 를 함께 해석한다.

In [ ]:
below = (score <= 0).mean()

print(f"  E[X]         = {EX:.2f}")
print(f"  sigma_X      = {sdX:.4f}")
print(f"  0점 이하 비율 = {below:.4f}   ({below*100:.1f}%)")

**"평균이 양수니까 찍으면 이득"** 이라는 말이 왜 불충분한가? 2~3문장으로.

> **답:** (여기에 작성)

---

## ✅ 제출 전 점검

- [ ] 맨 위에 **학번**을 적었다
- [ ] `TODO 1~5` 를 모두 채웠다
- [ ] 문제 1 계산 과정 · 2(c)(d) · 4(a)(b)(c) 의 **서술을 작성했다**
- [ ] `런타임 → 모두 실행` 으로 **모든 출력이 남아 있다**
- [ ] 그래프 두 장에 축 이름 · 제목 · 범례가 있고 **한글이 없다**
- [ ] 파일명을 **`HW_학번_T03.ipynb`** 로 바꿨다

**제출처와 기한은 PLATO · Google Classroom 공지를 확인한다.**

### 자주 하는 실수

- $\mathbb{E}[X_i^2]$ 를 $(\mathbb{E}[X_i])^2$ 로 계산 → **LOTUS 를 쓸 것**
- 분산을 구할 때 $\sigma$ 를 20배 → **분산이 20배**이고 $\sigma$ 는 $\sqrt{20}$ 배다
- 그래프 축 라벨을 한글로 → Colab 에서 깨진다